In [1]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
import SpaSRL as sp
warnings.filterwarnings("ignore")
import numpy as np
import random

np.random.seed(42)
random.seed(42)

In [2]:

def save_file(adata, results_path):
    df = pd.DataFrame(adata.obs['leiden'],columns=['Id','Expected'])

    for i in range(len(adata.obs['in_tissue'])):
        df = pd.concat((df,pd.DataFrame({"Id":[adata.obs_names[i]],"Expected":[adata.obs['leiden'][i]]})))
    df=df.set_index('Id')

    df.to_csv(results_path)
    

In [ ]:

method_name = "SpaSRL"
path = "/data/Ajita/Spatial/Datasets/Spatial_Domain/Real_datasets/mouse_breast_cancer/mouse_breast_cancer.h5ad"

results_path = "/data/Ajita/Spatial/Datasets/Spatial_Domain/Real_datasets/mouse_breast_cancer/final_domain_outputs/" \
+ "/output_" + method_name + ".csv"

adata = sc.read_h5ad(path)
print (adata.shape)

adata.var_names_make_unique()

sc.pp.highly_variable_genes(adata, n_top_genes=2000,flavor='seurat_v3',inplace=True)
sc.pp.log1p(adata)

adata = adata[:,adata.var['highly_variable']]

sp.spatial_enhancement(adata, alpha=1, n_neighbors=30, n_pcs=30, use_highly_variable=True, normalize_total=True, copy=False)

sp.run_SRL(adata,Lambda=0.1,n_neighbors=50)

res = 1.2
sc.tl.leiden(adata, resolution=res, neighbors_key='representation')
sp.refine_spatial_domains(adata, domain_key='leiden')

sc.tl.umap(adata, neighbors_key='representation')

sc.pl.embedding(adata, basis = "spatial", color = "leiden")
save_file(adata, results_path)


(1950, 32285)


 11%|██████████▋                                                                                      | 55/500 [00:28<00:30, 14.36it/s, relChg: 5.341e-05, recErr: 3.817e-01]

In [ ]:
res = 1.2 # increase to increase no of clusters = 0.025 default, 0.02=11, 

sc.tl.leiden(adata, resolution=res, neighbors_key='representation')
sc.pl.embedding(adata, basis = "spatial", color = "leiden")
